# 09 · HDFS + YARN Bootstrap (Caso C)

**Teoria**: docs/07-spark-e-hdfs.md

**Pré-requisito**: `make up-hadoop` (NameNode, 2 DataNodes, ResourceManager,
2 NodeManagers, e o gateway HttpFS — ~6GB de RAM recomendados).

---

🎯 **Objetivo**: este é o primeiro notebook onde o **Driver** (este processo, no seu
host) submete trabalho a um **gerenciador de cluster distribuído real (YARN)**, não ao
gerenciador Standalone do próprio Spark.

💡 O **YARN** (Yet Another Resource Negotiator) é o componente do ecossistema Hadoop
responsável por gerenciar recursos (CPU, memória) do cluster e agendar aplicações.
Diferente do modo Standalone do Spark, o YARN permite que múltiplos frameworks
(Spark, MapReduce, Flink) compartilhem o mesmo cluster simultaneamente.

📌 **Arquitetura em camadas que você verá:**
1. **Driver** (seu host) → submete a aplicação
2. **ResourceManager** (container `resourcemanager`) → aloca recursos
3. **NodeManagers** (`nodemanager1`/`nodemanager2`) → executam as Tasks
4. **HDFS** (NameNode + DataNodes) → armazenam os dados
5. **HttpFS Gateway** → ponte HTTP entre seu host e o HDFS

Leia docs/07 primeiro — em particular, por que usamos
`webhdfs://localhost:14000/...` aqui em vez do esquema `hdfs://namenode:8020/...`
que os slides de teoria mostram.

⚠️ **Atenção**: neste laboratório, o Driver executa no seu host (client mode),
mas **os executores executam dentro dos containers Docker** (`nodemanager1`/`nodemanager2`).
Isso significa que os executores NÃO têm acesso ao seu sistema de arquivos local
— por isso precisamos do HttpFS Gateway como ponte para transferir dados.

### 🧠 Como o YARN funciona (visão geral)

O YARN separa o **gerenciamento de recursos** do **framework de processamento**:

| Componente | Função | Container neste lab |
|---|---|---|
| **ResourceManager** | Gerencia recursos do cluster, mantém filas de aplicações | `resourcemanager` (porta 8088) |
| **NodeManager** | Executa tasks em um nó, reporta saúde ao RM | `nodemanager1`, `nodemanager2` |
| **ApplicationMaster** | Coordena a execução de uma aplicação específica | Iniciado dinamicamente em um NM |
| **Container** | Unidade de recursos (CPU+RAM) agendada pelo YARN | Onde as Tasks do Spark executam |

🔁 **Fluxo de submissão de uma aplicação Spark no YARN:**
1. Seu Driver (este notebook) conecta no ResourceManager via API REST
2. O ResourceManager aloca um container para o **ApplicationMaster**
3. O ApplicationMaster negocia containers para os **executores Spark**
4. Os executores rodam as Tasks e reportam resultados de volta ao Driver
5. Ao final, o ApplicationMaster libera os containers e a aplicação termina

> 💡 **Dica**: toda vez que você executar uma Action no Spark (`.count()`, `.show()`, `.write()`),
> você ativa todo esse pipeline — cada Job vira uma nova **Application** no YARN.
> Acompanhe em http://localhost:8088!

In [ ]:
import os
import sys
from pathlib import Path

# Adiciona o diretório scripts/ ao path do Python para importar funções auxiliares do laboratório
sys.path.insert(0, "../scripts")
from lab_utils import layer_path, upload_bronze_table_to_hdfs
from pyspark.sql import SparkSession

# Caso C: client mode contra um cluster YARN dockerizado (make up-hadoop).
# O Driver executa no seu HOST (este processo), mas os executores executam
# dentro dos containers nodemanager1/nodemanager2.

# HADOOP_CONF_DIR: informa ao Spark onde encontrar os arquivos XML de configuração
# do Hadoop (yarn-site.xml, core-site.xml) que contêm o endereço do ResourceManager.
# Sem esta variável, o Spark não sabe como conectar no YARN.
os.environ["HADOOP_CONF_DIR"] = str(Path("../config/hadoop-client").resolve())

# HADOOP_USER_NAME: como não há Kerberos ativo, o HDFS aceita qualquer usuário declarado.
# Os containers Docker rodam como root, então precisamos que o Driver se identifique
# como root também — senão o YARN negaria acesso ao diretório de staging.
os.environ["HADOOP_USER_NAME"] = "root"

# Cria a SparkSession apontando para o YARN como gerenciador de cluster
# .master("yarn") é a chave — sem isso, o Spark ignoraria o cluster e usaria local[*]
spark = (
    SparkSession.builder.appName("08-hdfs-yarn-bootstrap")
    .master("yarn")  # Usa YARN como cluster manager
    .config("spark.submit.deployMode", "client")  # Driver fica no HOST
    # spark.driver.host: essencial em client mode com Docker.
    # Os executores dentro dos containers precisam alcançar o Driver de volta.
    # docker-compose.yml mapeia host.docker.internal para o gateway do Docker.
    .config("spark.driver.host", "host.docker.internal")
    .config("spark.driver.bindAddress", "0.0.0.0")  # Aceita conexões de qualquer interface
    # Recursos dos executores — 2GB de RAM e 2 cores cada
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    # Memória do ApplicationMaster (o coordenador da aplicação dentro do YARN)
    .config("spark.yarn.am.memory", "1g")
    # spark.yarn.jars: aponta para os JARs do Spark já montados nos containers
    # (docker-compose.yml bind-mounts /opt/spark-jars em cada nodemanager).
    # Isso evita que o YARN tenha que fazer upload dos JARs para o HDFS.
    .config("spark.yarn.jars", "local:/opt/spark-jars/*")
    .getOrCreate()
)
spark  # Exibe a SparkSession configurada para confirmar

### ✅ SparkSession configurada para YARN — recap

📌 **Resumo das configurações que acabamos de definir:**

| Configuração | Valor | Propósito |
|---|---|---|
| `.master("yarn")` | `yarn` | Usar YARN como gerenciador de cluster |
| `spark.submit.deployMode` | `client` | Driver no HOST, executores nos containers |
| `spark.driver.host` | `host.docker.internal` | Executores alcançarem o Driver via Docker |
| `spark.executor.memory` | `2g` | 2GB de RAM por executor |
| `spark.executor.cores` | `2` | 2 tasks paralelas por executor |
| `spark.yarn.jars` | `local:/opt/spark-jars/*` | Evita upload desnecessário de JARs para HDFS |

> 💡 **Dica de diagnóstico**: se o Spark não conseguir conectar ao YARN, verifique:
> 1. `make status` — todos os containers estão RUNNING?
> 2. http://localhost:8088 — o ResourceManager responde?
> 3. `echo $HADOOP_CONF_DIR` — aponta para `config/hadoop-client`?
>
> ⚠️ O erro mais comum é o ResourceManager recusar a conexão porque os containers
> ainda estão inicializando. Aguarde ~30s após `make up-hadoop`.

## Sua primeira aplicação YARN

Até agora, todo Job que você executou usou `local[*]` (processo único) ou o Spark
Standalone (mestre+worker). Esta é a **primeira vez** que você submete trabalho
a um cluster YARN real!

🎯 **O que fazer antes de executar:**
1. Abra http://localhost:8088 (UI do ResourceManager) em uma aba do navegador
2. Observe a lista de Applications — deve estar vazia
3. Execute a célula abaixo
4. Durante a execução, volte à UI e veja a Application aparecer e transicionar entre estados

📌 **Estados que você verá na UI:**
- `ACCEPTED` → o ResourceManager recebeu a submissão e está alocando recursos
- `RUNNING` → o ApplicationMaster foi iniciado, executores estão rodando
- `FINISHED` → o Job terminou, containers foram liberados

⚠️ **Atenção**: esta célula pode demorar mais que o normal (10-20s extras). O YARN precisa
iniciar JVMs para os executores dentro dos containers antes de processar — isso é o
**overhead de cold start** do cluster, que não existe no modo local.

### 🎯 O que vamos executar?

Vamos criar um DataFrame com 5 milhões de números e simplesmente **contar** quantas
linhas existem:

```python
df = spark.range(0, 5_000_000)  # Cria uma coluna "id" com valores de 0 a 4.999.999
print(df.count())                # Action que dispara o Job no YARN
```

🧠 **Por que isso é especial aqui?** Embora `.range()` e `.count()` pareçam triviais,
esta é a primeira execução onde o Spark **negocia recursos com o YARN**, aloca
containers em NodeManagers, e executa tasks distribuídas.

> 💡 Compare o tempo deste `.count()` com o mesmo comando em `local[*]` (Lab 01).
> A diferença de tempo é o **custo da coordenação distribuída**: inicialização de
> containers, serialização de tasks, conexões de rede entre os containers.

In [ ]:
# Cria um DataFrame com 5 milhões de linhas (coluna "id" de 0 a 4.999.999)
df = spark.range(0, 5_000_000)

# .count() é uma ACTION no Spark — força a execução imediata de um Job
# Este Job vai: negociar containers com YARN → iniciar executores → distribuir partitions → contar
print(f"Row count: {df.count():,}")

print("Check http://localhost:8088 for the Application that just ran.")

### 📌 Análise dos resultados

Se você acompanhou a UI do ResourceManager (http://localhost:8088), deve ter observado:

1. ✅ **ACCEPTED** — O ResourceManager recebeu sua aplicação e a colocou na fila
2. ✅ **RUNNING** — O ApplicationMaster foi alocado em `nodemanager1` ou `nodemanager2`
3. ✅ **FINISHED** — O Job terminou, containers foram liberados para o cluster

💡 **Explore os detalhes na UI**: clique no ID da Application:
- Na aba **Containers**: quantos foram usados? (idealmente: 1 AM + N executores)
- Em **Logs**: veja a saída do `print()` do Driver (se disponível)
- Em **Tracking UI**: link para a Spark UI da aplicação

🧠 **Compare com sua experiência anterior:**
- Lab 01 (`local[*]`): sem YARN — tudo em um único processo, sem UI de ResourceManager
- Caso B (Standalone): Master e Workers próprios do Spark, UI na porta 8080
- **Agora (YARN)**: ResourceManager próprio na porta 8088, integração com HDFS

> 📌 **Cada modo de deploy tem seu propósito**: local é para desenvolvimento/teste,
> Standalone para ambientes exclusivos Spark, YARN para clusters multi-framework
> (Spark + MapReduce + Flink + ... no mesmo hardware).

## Enviando a camada Bronze para o HDFS

Diferente dos Casos B/D (onde um volume Docker compartilhado já torna `./data`
visível aos containers), os NodeManagers do Caso C **não têm acesso** ao seu
sistema de arquivos host.

🔁 **Por isso usamos o HttpFS Gateway como ponte:**
```
Seu host (arquivos Parquet em ./data/bronze/)
    ↓ chamadas REST HTTP (PUT via WebHDFS API)
HttpFS Gateway (container httpfs, porta 14000)
    ↓ chamadas RPC HDFS internas (dentro da rede Docker)
NameNode + DataNodes (HDFS)
    ↓ blocos replicados (fator de replicação 2)
/datalake/bronze/vendas/ (acessível por webhdfs://)
```

O helper `upload_bronze_table_to_hdfs()` percorre todos os arquivos Parquet
gerados localmente e os escreve no HDFS via chamadas REST **WebHDFS**, preservando
a estrutura de pastas de partição `ano=/mes=`.

📌 **Por que HttpFS e não HDFS diretamente?** Veja docs/07 — o protocolo nativo
`hdfs://` exige que o cliente esteja na mesma rede dos DataNodes. O HttpFS expõe
uma porta HTTP (14000) que o HOST pode acessar via `localhost`, e internamente
se comunica com o HDFS via RPC.

⚠️ **O upload pode levar alguns minutos** na primeira execução, especialmente para
a tabela `vendas` que tem muitas partições. É uma operação única — uma vez no HDFS,
os dados persistem entre execuções do notebook.

### 🔄 O que a função `upload_bronze_table_to_hdfs` faz?

Ela implementa o seguinte pipeline para cada tabela:
1. Lista os arquivos Parquet em `./data/bronze/<tabela>/`
2. Para cada arquivo, verifica se já existe no HDFS (operação idempotente)
3. Se não existir, faz upload via requisição REST:
   `PUT http://localhost:14000/webhdfs/v1/datalake/bronze/<tabela>/<arquivo>?op=CREATE`
4. O HttpFS recebe o conteúdo e escreve no HDFS via protocolo nativo

> 💡 **Isto é equivalente ao que a CLI `hdfs dfs -put` faz nos bastidores**
> — a diferença é que estamos usando a API REST diretamente do Python,
> sem precisar do cliente Hadoop instalado no host.

In [ ]:
# Faz upload dos dados Parquet locais para o HDFS via gateway HttpFS
# Lê os arquivos de ./data/bronze/vendas/ e escreve em /datalake/bronze/vendas/ no HDFS
upload_bronze_table_to_hdfs("vendas")
# Mesmo processo para empresas e funcionarios — cada tabela vira uma pasta separada no HDFS
upload_bronze_table_to_hdfs("empresas")
upload_bronze_table_to_hdfs("funcionarios")

### ✅ Upload concluído — dados no HDFS

Os dados agora residem no HDFS. Vamos lê-los de volta para confirmar a integridade.

🧠 **O fluxo de leitura será:**
1. O Driver faz uma requisição HTTP para `webhdfs://localhost:14000/datalake/bronze/vendas`
2. O HttpFS Gateway consulta o NameNode para localizar os blocos
3. O HttpFS lê os blocos dos DataNodes e retorna via HTTP
4. O Driver recebe os dados e os distribui para os executores no YARN
5. Os executores processam as partitions em paralelo nos containers Docker

📌 **Implicação importante**: cada byte trafega pela rede **três vezes**
(DataNode → HttpFS → Driver → Executor). Isso é inerentemente mais lento que
leitura local, mas é a arquitetura de um cluster real que separa armazenamento
e processamento.

In [ ]:
# Lê os dados do HDFS via webhdfs:// — o Spark monta o caminho automaticamente
# layer_path("hdfs", "bronze", "vendas") resolve para:
# webhdfs://localhost:14000/datalake/bronze/vendas
vendas_hdfs = spark.read.parquet(layer_path("hdfs", "bronze", "vendas"))

# .count() força a leitura de TODOS os arquivos — dispara outra Application no YARN!
# Desta vez os dados vêm do HDFS, não de um gerador interno (range)
print(f"Read back from HDFS: {vendas_hdfs.count():,} rows")

# .show(5) exibe as primeiras 5 linhas para confirmar integridade dos dados
# O Spark aplica projeção automática (lê apenas as colunas necessárias)
vendas_hdfs.show(5)

### 📌 Observação sobre a leitura do HDFS

Compare o número de linhas com a contagem do Lab 01 (quando os dados estavam no
volume local). **Os números devem ser idênticos** — os dados são os mesmos,
apenas o meio de armazenamento mudou.

✅ **Se os dados aparecerem corretamente, você confirmou que:**
1. ✅ O upload via HttpFS funcionou corretamente
2. ✅ A configuração `HADOOP_CONF_DIR` aponta para o lugar certo
3. ✅ O Spark consegue ler do HDFS usando o protocolo `webhdfs://`
4. ✅ Os executores no YARN estão recebendo dados através do Driver
5. ✅ As partições `ano=`/`mes=` foram preservadas na estrutura do HDFS

> 💡 **Dica**: explore a UI do NameNode em http://localhost:9870 → Utilities →
> Browse the file system. Navegue até `/datalake/bronze/vendas/` e veja os arquivos
> `part-*.parquet`. Cada arquivo corresponde a uma partition dos dados.
>
> 📌 Note o fator de replicação 2: cada bloco existe em **2 DataNodes**.
> Isso significa tolerância a falhas de armazenamento — se um DataNode cair,
> o HDFS ainda consegue servir os dados a partir da réplica.

## Verificação visual: a UI do NameNode

Para confirmar visualmente que os dados estão no HDFS:

1. Abra http://localhost:9870 no navegador
2. Vá em **Utilities** → **Browse the file system**
3. Navegue até `/datalake/bronze/vendas`
4. Você verá os diretórios de partição `ano=`/`mes=` e dentro deles os arquivos `part-*.parquet`

📌 **O que observar:**
- **Fator de replicação**: definido como 2 em `.env` — cada bloco existe em 2 DataNodes
- Se um DataNode falhar, o HDFS automaticamente usa a réplica do outro
- O Spark pode otimizar a leitura escolhendo o DataNode mais próximo (data locality)
- O custo de armazenamento é 2× o tamanho real (trade-off: tolerância a falhas vs. custo)

> 💡 **Compare com o RustFS (Lab 11)**: enquanto o HDFS replica blocos inteiros (fator 2),
> o RustFS usa **Erasure Coding RS(4,2)** — fragmenta os dados em 6 partes, onde
> qualquer 4 reconstroem o original. Mais eficiente em armazenamento (1.5× vs 2×).

📌 **Para encerrar**, vamos parar a SparkSession e liberar os recursos no YARN:

In [ ]:
# Finaliza a SparkSession, liberando todos os recursos no YARN
# Os containers dos executores serão encerrados e a memória/CPU devolvida ao cluster
spark.stop()